In [0]:
# 1) IMPORTS & LOGGING
import json
import logging
import requests
import yaml
from string import Template
from datetime import datetime, date
from typing import List, Dict, Tuple
from requests.adapters import HTTPAdapter, Retry
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, lit, current_timestamp
from pyspark.sql.utils import AnalysisException
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType



In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logging.getLogger("pyspark").setLevel(logging.WARNING)
logger = logging.getLogger("compass.apple")

storage_account_name = "compassdataprod"
container = "sa-compasslake"
secret_scope_name = "adlsscpkeydata"
secret_key_name   = "adlsstoragekeydata"

# Recupera o SAS Token do secret scope
sas_token = dbutils.secrets.get(scope=secret_scope_name, key=secret_key_name)

# Configuração com SAS Token
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "SAS")
spark.conf.set(f"fs.azure.sas.token.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.sas.FixedSASTokenProvider")
spark.conf.set(f"fs.azure.sas.fixed.token.{storage_account_name}.dfs.core.windows.net", sas_token)


In [0]:
# Variaveis de entrada
directory_application = "d_ingest_apple_store"
layer_source = "raw"  #raw, b (bronze), s (silver) ou g (gold)
format="csv"
delimiter=","
header=True
compression=""
date_partition_path="2025-08-25"
target_directory="b_compass/"
target_format="delta"
target_mode="overwrite"
target_partitionBy=["ref_date"]
control_table="b_compass"
create_empty_if_missing=True,
application_control="data_params"
application = "apple_reviews"

In [0]:
def get_config_template(
    version_config_file: str,
    
    # Fonte
    container: str,
    layer_source: str,
    storage_account: str,
    directory_application: str,
    format: str,
    delimiter: str,
    header: bool,
    compression: str,
    date_partition_path: str,
    
    # Destino
    target_directory: str,
    target_format: str,
    target_mode: str,
    target_partitionBy: list,
    
    # Controle
    control_table: str,
    control_description: str,
    application: str,
    
    # Fallback
    create_empty_if_missing: bool,
    
    # Schema
    application_control: str
) -> dict:
    """Retorna a configuração equivalente ao YAML, com telemetria como estrutura vazia."""
    
    config = {
        "version_config_file": version_config_file,
        
        "source": {
            "path_blob": f"abfss://{container}@{storage_account}.dfs.core.windows.net/",
            "layer_source": layer_source, #raw, b (bronze), s (silver) ou g (gold)
            "directory": f"{layer_source}_compass/{directory_application}/{date_partition_path}/",
            "format": format,
            "delimiter": delimiter,
            "header": header,
            "encoding": "utf-8",
            "compression": compression,
            "date_partition_column": date_partition_path
        },
        
        "target": {
            "path_blob": f"abfss://{container}@{storage_account}.dfs.core.windows.net/",
            "directory": target_directory,
            "format": target_format,
            "mode": target_mode,
            "partitionBy": target_partitionBy
        },
        
        "control": {
            "database": "b_compass",
            "table": control_table,
            "version": version_config_file,
            "description": control_description,
            "application": application
        },
        
        # Telemetria ainda não preenchida
        "telemetry": {"active": None},
        
        "default_telemetry": {          
            "owner": {"sigla": None, "projeto": None, "layer_lake": None},
            "valid_data": {"count": None, "percentage": None},
            "invalid_data": {"count": None, "percentage": None},
            "total_records": None,
            "total_processing_time": None,
            "validation_results": None,
            "success_count": None,
            "error_count": None,
            "type_client": None,
            "source": {"app": None, "search": None},
            "_ts": {"compass_start_ts": None, "compass_end_ts": None},
            "others_metrics": None,
            "timestamp": None
        },
        
        "fallback": {"create_empty_if_missing": create_empty_if_missing},
        
        "schema": {"control_database": "control_params_compass", "control_table": "data_params" ,"control_filter_reference": application_control}
    }
    
    return config


def fill_telemetry(config: dict, telemetry_values: dict) -> dict:
    """
    Preenche a estrutura de telemetria dentro do dicionário de configuração.
    :param config: dicionário retornado por get_config_template
    :param telemetry_values: dicionário com os valores a preencher
    :return: config atualizado com telemetria preenchida
    """
    # Preenche telemetria ativa
    if "telemetry_active" in telemetry_values:
        config["telemetry"]["active"] = telemetry_values.pop("telemetry_active")
    
    # Preenche default_telemetry
    for key, value in telemetry_values.items():
        if key in config["default_telemetry"]:
            config["default_telemetry"][key] = value
    
    return config

def func_read_source(
        path: str,
        format: str = "csv",
        delimiter: str = ",",
        header: bool = True,
        encoding: str = "utf-8",
        compression: str = None,
        date_partition_path: str = None,
        **kwargs
    ) -> DataFrame:
    """
        Lê dados de diferentes formatos de arquivo e compactações, baseado na configuração fornecida.
        
        :param path: Caminho completo do arquivo ou diretório
        :param format: Formato do arquivo: csv, parquet, avro, delta
        :param delimiter: Separador CSV (apenas para CSV)
        :param header: Se CSV tem cabeçalho
        :param encoding: Encoding do arquivo
        :param compression: Tipo de compactação: gzip, zip, snappy, None
        :param date_partition_column: Coluna de partição por data (opcional)
        :param kwargs: Parâmetros adicionais para Spark read
        :return: DataFrame Spark
    """
    
    reader = spark.read.format(format)
    
    if format.lower() == "csv":
        reader = reader.option("delimiter", delimiter)\
                       .option("header", header)\
                       .option("encoding", encoding)
        if compression:
            reader = reader.option("compression", compression)
    
    elif format.lower() == "parquet" and compression:
        reader = reader.option("compression", compression)
    
    elif format.lower() == "avro" and compression:
        reader = reader.option("compression", compression)
    
    # Aplicar opções extras
    for k, v in kwargs.items():
        reader = reader.option(k, v)
    
    try:
        df = reader.load(path)
    except AnalysisException as e:
        raise RuntimeError(f"Erro ao ler arquivos em {path}: {e}")
    
    # Checar se há dados sem forçar leitura de todo conteúdo
    # Spark otimiza .isEmpty() sem trazer todos os dados para driver
    if df.rdd.isEmpty():
        raise ValueError(f"Nenhum dado encontrado no caminho {path}")
    
    return df

In [0]:

config = get_config_template(
    version_config_file="v1.0",

    container=container,
    layer_source=layer_source,
    storage_account=storage_account_name,
    directory_application=directory_application,
    format=format,
    delimiter=delimiter,
    header=header,
    compression=compression,
    date_partition_path=date_partition_path,
    
    target_directory=target_directory,
    target_format=target_format,
    target_mode=target_mode,
    target_partitionBy=target_partitionBy,
    application=application,
    
    control_table=control_table,
    control_description="Bronze layer app_reviews reviews",
    
    create_empty_if_missing=create_empty_if_missing,
    application_control=application_control
)


source_config = config["source"]
source_config["path"] = source_config.pop("path_blob") + source_config.pop("directory")

df = func_read_source(**source_config)

display(df)



In [0]:
def validate_schema_types(df: DataFrame, layer: str, application: str, control_table: str):
    """
    Valida o schema de um DataFrame comparando nomes de colunas e tipos
    com o schema esperado registrado na tabela de contratos.

    :param df: DataFrame a validar
    :param layer: Camada de processamento (ex: raw, bronze)
    :param application: Nome da aplicação (ex: apple_reviews)
    :param control_table: Tabela Delta que guarda os contratos de schema
    :raises RuntimeError: Se o schema não corresponder ao esperado
    :return: None (apenas valida)
    """
    # Recupera o schema esperado mais recente da tabela de contratos
    latest_schema_row = (
        spark.table(control_table)
        .filter((F.col("layer") == layer) & (F.col("application") == application))
        .orderBy(F.desc("version"))
        .select("schema_expected_json")
        .limit(1)
        .collect()
    )

    if not latest_schema_row:
        raise RuntimeError(f"Nenhum schema encontrado para layer={layer}, application={application}")

    schema_expected_json = latest_schema_row[0]["schema_expected_json"]
    expected_schema = {col["name"]: col["type"].upper() for col in json.loads(schema_expected_json)["columns"]}

    # Schema atual do DataFrame
    current_schema = {field.name: field.dataType.simpleString().upper() for field in df.schema.fields}

    # Comparação: verifica divergências entre esperado e atual
    mismatches = {
        col: {"expected": expected_schema[col], "current": current_schema.get(col)}
        for col in expected_schema
        if col not in current_schema or current_schema[col] != expected_schema[col]
    }

    if mismatches:
        raise RuntimeError(
            f"Schema mismatch detected for layer={layer}, application={application}. Divergent columns: {json.dumps(mismatches, indent=2)}"
        )

    print(f"Schema validation passed for layer={layer}, application={application}.")

# Extrai diretamente os valores do config de forma segura
schema_config = config.get("schema", {})

# Chama a função passando o controle completo já formatado
validate_schema_types(
    df=df,
    layer=layer_source,
    application=application,
    control_table=f"{schema_config.get('control_database')}.{schema_config.get('control_table')}"
)